In [ ]:
!pip install streamlit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
!wget -q -O - ipv4.icanhazip.com

35.226.29.159


In [ ]:
%%writefile app.py
import numpy as np
import librosa
import streamlit as st
from scipy.signal import hilbert
from scipy.fft import fft, fftfreq
import plotly.graph_objects as go

# Constants
FREQ_MIN = 20
FREQ_MAX = 2000
FRAME_OFFSET = 2048
FFT_WINDOW_SIZE = 1024

# Indian Musical Notes frequencies
NOTES = [
    ("Sa", 261.63),  # C4
    ("Re", 293.66),  # D4
    ("Ga", 329.63),  # E4
    ("Ma", 349.23),  # F4
    ("Pa", 392.00),  # G4
    ("Dha", 440.00),  # A4
    ("Ni", 493.88),  # B4
    ("Sa", 523.25),  # C5
]

def find_nearest_note(frequency):
    if frequency <= 0:
        return "Silence"
    closest_note = min(NOTES, key=lambda note: abs(note[1] - frequency))
    return closest_note[0]

def freq_to_number(frequency):
    return 69 + 12 * np.log2(frequency / 440.0)

def note_name(note_number):
    note_names = ['sa', 'ko re', 're', 'ko ga', 'ga', 'ma', 'tr ma', 'pa', 'ko dha', 'dha', 'ko ni', 'ni']
    return note_names[int(round(note_number) % 12)]

def plot_fft(p, xf, notes, dimensions=(960, 540)):
    layout = go.Layout(
        title="Frequency Spectrum",
        autosize=False,
        width=dimensions[0],
        height=dimensions[1],
        xaxis_title="Frequency (Hz)",
        yaxis_title="Magnitude",
        font={'size': 24}
    )

    fig = go.Figure(layout=layout,
                    layout_xaxis_range=[FREQ_MIN, FREQ_MAX],
                    layout_yaxis_range=[0, 1])

    fig.add_trace(go.Scatter(x=xf, y=p))

    for note in notes:
        fig.add_annotation(x=note[0] + 10, y=note[2],
                           text=note[1],
                           font={'size': 16},
                           showarrow=False)
    return fig

def extract_sample(audio, frame_number):
    end = frame_number * FRAME_OFFSET
    begin = int(end - FFT_WINDOW_SIZE)

    if end == 0:
        return np.zeros((np.abs(begin)), dtype=float)
    elif begin < 0:
        return np.concatenate([np.zeros((np.abs(begin)), dtype=float), audio[0:end]])
    else:
        return audio[begin:end]

def find_top_notes(fft, xf, num):
    if np.max(np.abs(fft)) < 0.001:
        return []

    lst = sorted(enumerate(np.abs(fft)), key=lambda x: x[1], reverse=True)

    idx = 0
    found = []
    found_note = set()

    while idx < len(lst) and len(found) < num:
        fft_idx = lst[idx][0]

        if fft_idx >= len(xf):
            idx += 1
            continue

        f = xf[fft_idx]
        y = lst[idx][1]
        n = freq_to_number(f)
        n0 = int(round(n))
        name = note_name(n0)

        if name not in found_note:
            found_note.add(name)
            found.append([f, name, y])

        idx += 1

    return found

def process_audio(audio_data, sample_rate, segment_duration, num_top_notes=3):
    samples_per_segment = int(segment_duration * sample_rate)
    results = []

    for i in range(0, len(audio_data), samples_per_segment):
        segment = audio_data[i:i + samples_per_segment]

        if len(segment) == samples_per_segment:
            fft_result = fft(segment)
            freqs = fftfreq(len(segment), 1 / sample_rate)
            magnitudes = np.abs(fft_result)
            positive_freqs = freqs[:len(freqs) // 2]
            positive_magnitudes = magnitudes[:len(magnitudes) // 2]

            # Find top notes from the FFT result
            top_notes = find_top_notes(fft_result, positive_freqs, num_top_notes)

            # Include timestamp for each detected note
            timestamp = i / sample_rate
            for note in top_notes:
                results.append([timestamp, note[1], note[0]])

    return results

# Streamlit UI
def main():
    st.title("Indian Classical Music Note Detection")

    uploaded_file = st.file_uploader("Upload an Audio File", type=["wav", "mp3"])
    if uploaded_file is not None:
        st.audio(uploaded_file, format="audio/wav")
        file_path = uploaded_file.name

        target_sample_rate = 22050
        audio_data, sample_rate = librosa.load(uploaded_file, sr=target_sample_rate)
        audio_data, _ = librosa.effects.trim(audio_data, top_db=20)

        segment_duration = st.slider("Segment Duration (seconds)", 0.1, 1.0, 0.1)
        num_top_notes = st.slider("Number of Top Notes to Extract", 1, 5, 3)

        results = process_audio(audio_data, sample_rate, segment_duration, num_top_notes)

        # Display results as a table
        if results:
            st.write("Detected Notes:")
            st.write("### Timestamp, Note, Frequency")
            st.dataframe(
                [{"Timestamp (s)": r[0], "Note": r[1], "Frequency (Hz)": round(r[2], 2)} for r in results]
            )
        else:
            st.write("No notes detected.")

        # Plot frequency spectrum with top notes
        magnitudes = np.abs(fft(audio_data))
        freqs = fftfreq(len(audio_data), 1 / sample_rate)
        positive_freqs = freqs[:len(freqs) // 2]
        positive_magnitudes = magnitudes[:len(magnitudes) // 2]

        plot_fig = plot_fft(positive_magnitudes, positive_freqs, results)
        st.plotly_chart(plot_fig)

if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴

⠦⠧⠇⠏⠋⠙Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.127.45.1:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://shy-wombats-guess.loca.lt
